In [0]:
# --- Configuration ---
bu_filter = "COMCE-Germany,Austria, Switzerl,CEE"

In [0]:
%sql
WITH Owned_Reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UP.BU IN ('COMCE-Germany,Austria, Switzerl,CEE')
)

-- Access-only documents (not owned by BU users)
SELECT
  'Only Access' AS category,
  COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
  ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
  AND UP.BU IN ('COMCE-Germany,Austria, Switzerl,CEE')
  AND cms.Document_Id NOT IN (SELECT DISTINCT Document_Id FROM Owned_Reports)

union all 
-- Access-own documents
SELECT
  'Access Owned' AS category,
  COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
  ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
  AND UP.BU IN ('COMCE-Germany,Austria, Switzerl,CEE')
  AND cms.Document_Id IN (SELECT DISTINCT Document_Id FROM Owned_Reports)
;

WITH Owned_Reports AS (
  SELECT DISTINCT
    cms.Document_Id,
    CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UP.BU IN ('COMCE-Germany,Austria, Switzerl,CEE')
)
-- Owned documents without schedule
SELECT
  'Total owned normal reports' AS category,
  COUNT(DISTINCT Document_Id) AS Document_cnt
FROM Owned_Reports
WHERE scheduled = 'N'

UNION ALL

-- Owned documents with schedule
SELECT
  'Total owned scheduled reports' AS category,
  COUNT(DISTINCT Document_Id) AS Document_cnt
FROM Owned_Reports
WHERE scheduled = 'Y'

In [0]:
%sql
with owned as (
    select distinct cluster, Doc_ID from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis
    where OwnerBU = upper('COMCE-Germany,Austria, Switzerl,CEE')
)

select 'Accessed additionaly' as category, cluster, count(distinct Doc_id) as reports from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis 
where Doc_id not in (select Doc_ID from owned)
and BU = upper('COMCE-Germany,Austria, Switzerl,CEE')
and cluster is not null
group by cluster
union all 
select 'Owned' as category, cluster, count(distinct Doc_ID) as reports  from owned
where cluster is not null
group by cluster
order by category asc, cluster asc 

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Query 1: Document access vs ownership ---
pdf = spark.sql(f"""
  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UP.BU IN ('{bu_filter}')
  )

  SELECT
    'Only Access' AS category,
    COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UP.BU IN ('{bu_filter}')
    AND cms.Document_Id NOT IN (SELECT DISTINCT Document_Id FROM Owned_Reports)

  UNION ALL

  SELECT
    'Access Owned' AS category,
    COUNT(DISTINCT CMS.Document_Id) AS Document_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS UA
    ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(CAST(UA.WEBI_Doc_ID AS STRING)))
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(UP.UserName)) = UPPER(TRIM(UA.UserName))
  WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND UP.BU IN ('{bu_filter}')
    AND cms.Document_Id IN (SELECT DISTINCT Document_Id FROM Owned_Reports)
""").toPandas()

# --- Query 2: Owned reports - normal vs scheduled ---
pdf2 = spark.sql(f"""
  WITH Owned_Reports AS (
    SELECT DISTINCT
      cms.Document_Id,
      CASE WHEN S1.document_id IS NULL THEN 'N' ELSE 'Y' END AS scheduled
    FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
      ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
    LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted AS S1
      ON UPPER(TRIM(CMS.Document_Id)) = UPPER(TRIM(S1.document_id))
    WHERE UPPER(TRIM(cms.Document_name)) NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND UP.BU IN ('{bu_filter}')
  )

  SELECT
    'Total owned normal reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'N'

  UNION ALL

  SELECT
    'Total owned scheduled reports' AS category,
    COUNT(DISTINCT Document_Id) AS Document_cnt
  FROM Owned_Reports
  WHERE scheduled = 'Y'
""").toPandas()

# --- Combined Chart: Two stacked bars side by side (Navy Theme) ---
access_owned = pdf.loc[pdf['category'] == 'Access Owned', 'Document_cnt'].values[0]
only_access = pdf.loc[pdf['category'] == 'Only Access', 'Document_cnt'].values[0]
normal = pdf2.loc[pdf2['category'] == 'Total owned normal reports', 'Document_cnt'].values[0]
scheduled = pdf2.loc[pdf2['category'] == 'Total owned scheduled reports', 'Document_cnt'].values[0]

fig, ax = plt.subplots(figsize=(8, 7))

# Bar 1: Access distribution
ax.bar('Access\nDistribution', access_owned, color='#336699', label='Access Owned')
ax.bar('Access\nDistribution', only_access, bottom=access_owned, color='#FABE6E', label='Only Access')
ax.text(0, access_owned / 2, f'{access_owned:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
ax.text(0, access_owned + only_access / 2, f'{only_access:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')

# Bar 2: Owned reports breakdown
ax.bar('Owned Reports\nBreakdown', normal, color='#336699', label='Normal Reports')
ax.bar('Owned Reports\nBreakdown', scheduled, bottom=normal, color='#FFD700', label='Scheduled Reports')
ax.text(1, normal / 2, f'{normal:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='white')
ax.text(1, normal + scheduled / 2, f'{scheduled:,}', ha='center', va='center', fontweight='bold', fontsize=12, color='black')

ax.set_ylabel('Reports Count')
ax.set_title(f'{bu_filter} - Overview')
ax.legend(title='Category', fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Query: Accessed additionally vs Owned by cluster ---
pdf3 = spark.sql(f"""
  WITH owned AS (
    SELECT DISTINCT cluster, Doc_ID
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_Owner_analysis
    WHERE OwnerBU = UPPER('{bu_filter}')
  )

  SELECT 'Accessed additionally' AS category, cluster, COUNT(DISTINCT Doc_id) AS reports
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
  WHERE Doc_id NOT IN (SELECT Doc_ID FROM owned)
    AND BU = UPPER('{bu_filter}')
    AND cluster IS NOT NULL
  GROUP BY cluster

  UNION ALL

  SELECT 'Owned' AS category, cluster, COUNT(DISTINCT Doc_ID) AS reports
  FROM owned
  WHERE cluster IS NOT NULL
  GROUP BY cluster
  ORDER BY category ASC, cluster ASC
""").toPandas()

# --- Grouped Bar Chart: reports by cluster, legend by category ---
pivot = pdf3.pivot(index='cluster', columns='category', values='reports').fillna(0)
clusters = pivot.index.astype(int).astype(str)
x = np.arange(len(clusters))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width/2, pivot.get('Accessed additionally', 0), width, color='#FABE6E', label='Accessed additionally')
bars2 = ax.bar(x + width/2, pivot.get('Owned', 0), width, color='#336699', label='Owned')

# Add value labels
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 5, f'{int(h):,}', ha='center', va='bottom', fontsize=9, color='black')
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 5, f'{int(h):,}', ha='center', va='bottom', fontsize=9, color='black')

ax.set_xlabel('Cluster')
ax.set_ylabel('Reports Count')
ax.set_title(f'{bu_filter} - Reports by Cluster')
ax.set_xticks(x)
ax.set_xticklabels(clusters)
ax.legend(title='Category', fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
%sql
WITH eventCategorized AS (
  SELECT
    ua1.*,
    CASE
      WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
      ELSE 'Viewer_events'
    END AS usage_category
  FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
    ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
  WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
    AND CMS.Document_Id IS NOT NULL
)

SELECT
  ec.UserName as User_ID, up1.DisplayName as User_Name, up1.BU as User_Bu, up1.Title as User_Role,
  SUM(CASE WHEN usage_category = 'Viewer_events' THEN 1 ELSE 0 END) AS Viewer_events,
  COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN WEBI_Doc_ID END) AS Viewer_report_cnt
FROM eventCategorized AS ec
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
  ON ec.UserName = up1.UserName
WHERE up1.BU = 'COMCE-Germany,Austria, Switzerl,CEE'
and up1.UserName not in ('DESWAH1','DEJSCH6','DECGAS2','DEMJAN2','CZLBEC1','DEIRAT2','CHPSTU1')
GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title
ORDER BY Viewer_report_cnt DESC
limit 7



In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Query: Top Editors in COMCE BU ---
pdf_editors = spark.sql(f"""
  WITH eventCategorized AS (
    SELECT
      ua1.*,
      CASE
        WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
        ELSE 'Viewer_events'
      END AS usage_category
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
      ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
    WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND CMS.Document_Id IS NOT NULL
  )

  SELECT
    ec.UserName AS User_ID,
    up1.DisplayName AS User_Name,
    up1.BU AS User_BU,
    up1.Title AS User_Role,
    SUM(CASE WHEN usage_category = 'Editor_events' THEN 1 ELSE 0 END) AS Editor_events,
    COUNT(DISTINCT CASE WHEN usage_category = 'Editor_events' THEN WEBI_Doc_ID END) AS Editor_report_cnt
  FROM eventCategorized AS ec
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON ec.UserName = up1.UserName
  WHERE up1.BU = '{bu_filter}'
  GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title
  ORDER BY Editor_report_cnt DESC
  LIMIT 7
""").toPandas()

# --- Build exclusion list from editors ---
editor_ids = pdf_editors['User_ID'].tolist()
editor_ids_sql = ",".join([f"'{uid}'" for uid in editor_ids])

# --- Query: Top Viewers (excluding editors) ---
pdf_viewers = spark.sql(f"""
  WITH eventCategorized AS (
    SELECT
      ua1.*,
      CASE
        WHEN event_type IN ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') THEN 'Editor_events'
        ELSE 'Viewer_events'
      END AS usage_category
    FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit AS ua1
    LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
      ON ua1.WEBI_Doc_ID = CAST(CMS.Document_Id AS STRING)
    WHERE CMS.Document_name NOT IN ('DOCUMENT NOT FOUND ON SERVER')
      AND CMS.Document_Id IS NOT NULL
  )

  SELECT
    ec.UserName AS User_ID,
    up1.DisplayName AS User_Name,
    up1.BU AS User_BU,
    up1.Title AS User_Role,
    SUM(CASE WHEN usage_category = 'Viewer_events' THEN 1 ELSE 0 END) AS Viewer_events,
    COUNT(DISTINCT CASE WHEN usage_category = 'Viewer_events' THEN WEBI_Doc_ID END) AS Viewer_report_cnt
  FROM eventCategorized AS ec
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON ec.UserName = up1.UserName
  WHERE up1.BU = '{bu_filter}'
    AND up1.UserName NOT IN ({editor_ids_sql})
  GROUP BY ec.UserName, up1.DisplayName, up1.BU, up1.Title
  ORDER BY Viewer_report_cnt DESC
  LIMIT 7
""").toPandas()

# --- Format numbers ---
pdf_editors['Editor_events'] = pdf_editors['Editor_events'].apply(lambda x: f'{x:,}')
pdf_editors['Editor_report_cnt'] = pdf_editors['Editor_report_cnt'].astype(int)
pdf_viewers['Viewer_events'] = pdf_viewers['Viewer_events'].apply(lambda x: f'{x:,}')
pdf_viewers['Viewer_report_cnt'] = pdf_viewers['Viewer_report_cnt'].astype(int)

# --- Figure 1: Top Editors ---
fig, ax = plt.subplots(figsize=(10, 5))
names = pdf_editors['User_Name'].fillna(pdf_editors['User_ID'])
y_pos = range(len(names))
ax.barh(y_pos, pdf_editors['Editor_report_cnt'], color='#336699')
ax.set_yticks(y_pos)
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('Distinct Reports Edited')
ax.set_title(f'{bu_filter} - Top 7 Editors by Report Count')
ax.invert_yaxis()
for i, v in enumerate(pdf_editors['Editor_report_cnt']):
    ax.text(v + 1, i, f'{v:,}', va='center', fontsize=10, fontweight='bold', color='#336699')
plt.tight_layout()
plt.show()

# --- Figure 2: Top Viewers (excluding editors) ---
fig2, ax2 = plt.subplots(figsize=(10, 5))
names2 = pdf_viewers['User_Name'].fillna(pdf_viewers['User_ID'])
y_pos2 = range(len(names2))
ax2.barh(y_pos2, pdf_viewers['Viewer_report_cnt'], color='#FABE6E')
ax2.set_yticks(y_pos2)
ax2.set_yticklabels(names2, fontsize=10)
ax2.set_xlabel('Distinct Reports Viewed')
ax2.set_title(f'{bu_filter} - Top 7 Viewers by Report Count (excl. Editors)')
ax2.invert_yaxis()
for i, v in enumerate(pdf_viewers['Viewer_report_cnt']):
    ax2.text(v + 1, i, f'{v:,}', va='center', fontsize=10, fontweight='bold', color='#996633')
plt.tight_layout()
plt.show()

# --- Display tables ---
pdf_editors_disp = pdf_editors.rename(columns={'User_ID': 'User ID', 'User_Name': 'Name', 'User_BU': 'Business Unit', 'User_Role': 'Role', 'Editor_events': 'Events', 'Editor_report_cnt': 'Reports'})
pdf_editors_disp.insert(0, 'Category', 'Editor')
pdf_viewers_disp = pdf_viewers.rename(columns={'User_ID': 'User ID', 'User_Name': 'Name', 'User_BU': 'Business Unit', 'User_Role': 'Role', 'Viewer_events': 'Events', 'Viewer_report_cnt': 'Reports'})
pdf_viewers_disp.insert(0, 'Category', 'Viewer')
pdf_combined = pd.concat([pdf_editors_disp, pdf_viewers_disp], ignore_index=True).drop(columns=['Business Unit', 'User ID'])
display(pdf_combined)

In [0]:
%sql
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email others'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Location'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE up1.BU = 'COMCE-Germany,Austria, Switzerl,CEE'
)

SELECT
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Location'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt,
  COUNT(DISTINCT CMS.created) AS Unique_Owner_cnt,
  ARRAY_JOIN(SORT_ARRAY(COLLECT_SET(UP.DisplayName)), ' | ') AS Report_Owners
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE UP.BU = 'COMCE-Germany,Austria, Switzerl,CEE'
  AND sd2.document_id IS NOT NULL
GROUP BY
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Location'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END

In [0]:
%sql
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email others'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Location'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE up1.BU = 'COMCE-Germany,Austria, Switzerl,CEE'
),

base AS (
  SELECT
    UP.DisplayName as User_name,
    CASE
      WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
      WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
      WHEN sd2.destination_type = 'filesystem' THEN 'Shared Location'
      WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
      ELSE sd2.destination_type
    END AS Category,
    COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN labled_schedule
    ON sd2.document_id = labled_schedule.document_id
  WHERE UP.BU = 'COMCE-Germany,Austria, Switzerl,CEE'
    AND sd2.document_id IS NOT NULL
  GROUP BY
    UP.DisplayName,
    CASE
      WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
      WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
      WHEN sd2.destination_type = 'filesystem' THEN 'Shared Location'
      WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
      ELSE sd2.destination_type
    END
),

base_with_total AS (
  SELECT * FROM base
  UNION ALL
  SELECT
    UP.DisplayName as User_name,
    'Total' AS Category,
    COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
    ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
  LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
    ON CAST(cms.Document_Id AS STRING) = sd2.document_id
  LEFT JOIN labled_schedule
    ON sd2.document_id = labled_schedule.document_id
  WHERE UP.BU = 'COMCE-Germany,Austria, Switzerl,CEE'
    AND sd2.document_id IS NOT NULL
  GROUP BY UP.DisplayName
)

SELECT *
FROM base_with_total
PIVOT (
  SUM(Reports_cnt)
  FOR Category IN (
    'SAP BO' AS SAP_BO,
    'Email self' AS Email_self,
    'Email others' AS Email_others,
    'Shared Location' AS Shared_Location,
    'Paused' AS Paused,
    'Total' AS Total
  )
)
ORDER BY User_name

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# --- Query: Schedule Report Owners ---
pdf_sched_owners = spark.sql(f"""
WITH labled_schedule AS (
  SELECT
    sd1.sent_to,
    up1.EmailAddress,
    CASE
      WHEN sd1.destination_type = 'mail' AND UPPER(sd1.sent_to) = UPPER(up1.EmailAddress) THEN 'Email self'
      WHEN sd1.destination_type = 'mail' THEN 'Email others'
      WHEN sd1.destination_type = 'filesystem' THEN 'Shared Location'
      ELSE NULL
    END AS Consumption,
    sd1.*
  FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details_enriched AS sd1
  LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS up1
    ON sd1.document_created_by = up1.UserName
  WHERE up1.BU = '{bu_filter}'
)
SELECT
  UP.DisplayName,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Location'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END AS Category,
  COUNT(DISTINCT CMS.Document_Id) AS Reports_cnt
FROM dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata AS CMS
LEFT JOIN dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile AS UP
  ON UPPER(TRIM(CMS.created)) = UPPER(TRIM(UP.UserName))
LEFT JOIN dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details AS sd2
  ON CAST(cms.Document_Id AS STRING) = sd2.document_id
LEFT JOIN labled_schedule
  ON sd2.document_id = labled_schedule.document_id
WHERE UP.BU = '{bu_filter}'
  AND sd2.document_id IS NOT NULL
GROUP BY
  UP.DisplayName,
  CASE
    WHEN sd2.destination_type = 'mail' AND labled_schedule.Consumption IS NOT NULL THEN labled_schedule.Consumption
    WHEN sd2.destination_type = 'inbox' OR sd2.destination_type IS NULL THEN 'SAP BO'
    WHEN sd2.destination_type = 'filesystem' THEN 'Shared Location'
    WHEN labled_schedule.Consumption IS NULL THEN 'Paused'
    ELSE sd2.destination_type
  END
ORDER BY Category ASC, Reports_cnt DESC
""").toPandas()

# --- Pivot for stacked bar chart ---
pivot = pdf_sched_owners.pivot(index='DisplayName', columns='Category', values='Reports_cnt').fillna(0)
owners = pivot.index
categories = pivot.columns
colors = {
    'Email self': '#336699',
    'Email others': '#FABE6E',
    'Shared Location': '#FFD700',
    'SAP BO': '#A3A3A3',
    'Paused': '#D9534F'
}

fig, ax = plt.subplots(figsize=(12, 6))
bottom = pd.Series([0]*len(owners), index=owners)
for cat in categories:
    ax.bar(owners, pivot[cat], bottom=bottom, color=colors.get(cat, '#888888'), label=cat)
    bottom += pivot[cat]

ax.set_ylabel('Scheduled Reports Count')
ax.set_title(f'{bu_filter} - Scheduled Report Owners by Category')
ax.legend(title='Schedule Category', fontsize=10, loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# --- Display table ---
display(pdf_sched_owners)